# Linear Autoencoder for Spatio-Temporal Data Compression

**Approach**: Temporal-point encoding: each spatial point's full temporal sequence is compressed into a compact latent vector via a fully connected autoencoder.

**Three model configurations** are trained and compared:

| Model | Encoder | Latent Dim | Decoder |
|-------|---------|------------|--------|
| **Base** | 1200 → 256 → 128 | 16 | 128 → 256 → 1200 |
| **Medium** | 1200 → 512 → 256 → 128 | 32 | 128 → 256 → 512 → 1200 |
| **Large** | 1200 → 512 → 256 → 128 | 64 | 128 → 256 → 512 → 1200 |

**Input dimension**: `num_timesteps x num_vars` = `300 x 4` = 1200 per spatial point  
**Compressed storage**: Model weights + one latent code vector per spatial point

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
# Data path: /kaggle/input/datasets/maheshsadupalli/ml-test-loader-original-data-csv/ML_test_loader_original_data.csv


In [ ]:
"""
Linear AutoEncoder: Utilities, Models, and Training Functions

This cell defines all classes and functions needed for training and
evaluating fully connected autoencoders on spatio-temporal field data.
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import pyarrow.csv as pv
from torcheval.metrics import PeakSignalNoiseRatio
from torchmetrics.image import StructuralSimilarityIndexMeasure
import matplotlib.pyplot as plt
import time
import os
import json


# Dataset

class TemporalPointDataset(Dataset):
    """
    Reshapes flat spatio-temporal CSV into per-point temporal sequences.

    Each sample contains all field variables across all timesteps for a
    single spatial point, flattened into a 1-D vector.

    Reshaping pipeline:
        Raw CSV  : (num_points * num_timesteps, 8)
        Per point: (num_timesteps, num_vars)  -->  flatten  -->  (num_timesteps * num_vars,)
        Dataset  : (num_points, num_timesteps * num_vars)
    """

    def __init__(self, filepath):
        print("Loading dataset from: {}".format(filepath))

        read_options = pv.ReadOptions(
            column_names=['x', 'y', 'z', 't', 'Vx', 'Vy', 'Pressure', 'TKE']
        )
        table = pv.read_csv(filepath, read_options=read_options)
        data = table.to_pandas()

        # Sort by spatial location then time for consistent grouping
        data = data.sort_values(['x', 'y', 'z', 't']).reset_index(drop=True)

        coords = data[['x', 'y', 'z']].values.astype(np.float32)
        self.time_values = np.sort(data['t'].unique()).astype(np.float32)
        fields = data[['Vx', 'Vy', 'Pressure', 'TKE']].values.astype(np.float32)

        self.num_timesteps = data['t'].nunique()
        self.num_points = len(data) // self.num_timesteps
        self.num_vars = 4
        self.var_names = ['Vx', 'Vy', 'Pressure', 'TKE']

        print("Detected {} spatial points across {} timesteps".format(
            self.num_points, self.num_timesteps))

        # Reshape: (N * T, V) --> (N, T, V)
        fields_3d = fields.reshape(self.num_points, self.num_timesteps, self.num_vars)

        # Min-max normalization to [0, 1]
        self.field_min = fields.min(axis=0)
        self.field_max = fields.max(axis=0)
        self.field_range = self.field_max - self.field_min
        self.field_range[self.field_range == 0] = 1.0
        fields_norm = (fields_3d - self.field_min) / self.field_range

        # Flatten temporal dimension: (N, T * V)
        self.input_dim = self.num_timesteps * self.num_vars
        self.data = torch.FloatTensor(fields_norm.reshape(self.num_points, -1))

        # Keep 3-D array for per-timestep visualisation
        self.data_3d = torch.FloatTensor(fields_norm)  # (N, T, V)

        # Unique spatial coordinates (one row per point)
        self.coords = coords.reshape(
            self.num_points, self.num_timesteps, 3
        )[:, 0, :]

        print("Dataset ready: {} samples, input dimension = {}".format(
            self.num_points, self.input_dim))

    def __len__(self):
        return self.num_points

    def __getitem__(self, idx):
        x = self.data[idx]
        return x, x  # autoencoder: input equals target

    def get_timestep_data(self, timestep_idx):
        """Return field data at a given timestep for all points. Shape: (N, V)."""
        return self.data_3d[:, timestep_idx, :]

    def get_normalization_params(self):
        """Return normalization parameters as a JSON-serializable dictionary."""
        return {
            'field_min': self.field_min.tolist(),
            'field_max': self.field_max.tolist(),
            'field_range': self.field_range.tolist(),
            'num_timesteps': self.num_timesteps,
            'num_points': self.num_points,
            'num_vars': self.num_vars,
            'input_dim': self.input_dim
        }


# Models

class LinearEncoder(nn.Module):
    """Fully connected encoder with progressive dimensionality reduction."""

    def __init__(self, input_dim, hidden_dims, latent_dim, dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.LeakyReLU(0.1),
                nn.Dropout(dropout),
            ])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, latent_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class LinearDecoder(nn.Module):
    """Fully connected decoder that reconstructs from the latent space."""

    def __init__(self, latent_dim, hidden_dims, output_dim, dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = latent_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.LeakyReLU(0.1),
                nn.Dropout(dropout),
            ])
            prev_dim = h_dim
        # No activation on the output layer (regression task)
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, z):
        return self.network(z)


class LinearAutoEncoder(nn.Module):
    """
    Fully connected autoencoder for spatio-temporal data compression.

    Compresses the temporal sequence of field variables at each spatial
    point into a compact latent representation.
    """

    def __init__(self, input_dim, encoder_hidden, decoder_hidden,
                 latent_dim, dropout=0.1):
        super().__init__()
        self.encoder = LinearEncoder(input_dim, encoder_hidden,
                                     latent_dim, dropout)
        self.decoder = LinearDecoder(latent_dim, decoder_hidden,
                                     input_dim, dropout)
        self.latent_dim = latent_dim
        self.input_dim = input_dim

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)


# Model Configurations

AE_CONFIGS = {
    'base': {
        'encoder_hidden': [256, 128],
        'decoder_hidden': [128, 256],
        'latent_dim': 16,
        'dropout': 0.1,
    },
    'medium': {
        'encoder_hidden': [512, 256, 128],
        'decoder_hidden': [128, 256, 512],
        'latent_dim': 32,
        'dropout': 0.1,
    },
    'large': {
        'encoder_hidden': [512, 256, 128],
        'decoder_hidden': [128, 256, 512],
        'latent_dim': 64,
        'dropout': 0.1,
    },
}


def create_autoencoder(size, input_dim):
    """Instantiate a LinearAutoEncoder from a named configuration."""
    cfg = AE_CONFIGS[size]
    return LinearAutoEncoder(
        input_dim=input_dim,
        encoder_hidden=cfg['encoder_hidden'],
        decoder_hidden=cfg['decoder_hidden'],
        latent_dim=cfg['latent_dim'],
        dropout=cfg['dropout'],
    )


# Metrics

def compute_psnr_ssim(predictions, targets, device):
    """Compute PSNR (dB) and SSIM for autoencoder reconstructions."""
    predictions = predictions.to(device)
    targets = targets.to(device)

    # PSNR
    psnr_metric = PeakSignalNoiseRatio().to(device)
    psnr_metric.update(predictions, targets)
    psnr = psnr_metric.compute().item()

    # SSIM: reshape to image-like format (1, 1, N, D)
    pred_ssim = predictions.unsqueeze(0).unsqueeze(0)
    target_ssim = targets.unsqueeze(0).unsqueeze(0)
    ssim_metric = StructuralSimilarityIndexMeasure(
        gaussian_kernel=False, kernel_size=1
    ).to(device)
    ssim_metric.update(pred_ssim, target_ssim)
    ssim = ssim_metric.compute().item()

    return psnr, ssim


def compute_relative_error(predictions, targets):
    """Compute relative L2 norm error as a percentage."""
    error_norm = torch.norm(predictions - targets)
    target_norm = torch.norm(targets)
    return (error_norm / target_norm * 100).item()


def compute_compression_ratio(model, dataset):
    """
    Compute compression statistics.

    Compressed size = model weights (float32) + latent codes (float32).
    """
    total_params = sum(p.numel() for p in model.parameters())
    model_bytes = total_params * 4
    latent_bytes = dataset.num_points * model.latent_dim * 4
    compressed = model_bytes + latent_bytes
    original = dataset.num_points * dataset.num_timesteps * dataset.num_vars * 4
    return {
        'total_params': total_params,
        'model_size_kb': model_bytes / 1024,
        'latent_size_kb': latent_bytes / 1024,
        'compressed_size_kb': compressed / 1024,
        'original_size_mb': original / (1024 ** 2),
        'compression_ratio': original / compressed,
    }


# Training

def train_autoencoder(model, train_loader, dataset, device, num_epochs,
                      model_name, output_dir):
    """
    Train the autoencoder and save model, latent codes, normalization
    parameters, and per-epoch metrics.

    Returns:
        dict: Training metrics keyed by metric name.
    """
    os.makedirs(output_dir, exist_ok=True)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    metrics = {
        'loss': [],
        'psnr': [],
        'ssim': [],
        'relative_error': [],
        'time_per_epoch': [],
    }

    comp = compute_compression_ratio(model, dataset)
    total_params = comp['total_params']

    print("")
    print("=" * 65)
    print("  Training: {}".format(model_name))
    print("=" * 65)
    print("  Spatial points     : {:,}".format(dataset.num_points))
    print("  Timesteps          : {}".format(dataset.num_timesteps))
    print("  Input dimension    : {} ({} x {})".format(
        dataset.input_dim, dataset.num_timesteps, dataset.num_vars))
    print("  Latent dimension   : {}".format(model.latent_dim))
    print("  Parameters         : {:,}".format(total_params))
    print("  Model size         : {:.2f} KB".format(comp['model_size_kb']))
    print("  Latent storage     : {:.2f} KB".format(comp['latent_size_kb']))
    print("  Total compressed   : {:.2f} KB".format(comp['compressed_size_kb']))
    print("  Compression ratio  : {:.1f}:1".format(comp['compression_ratio']))
    print("  Epochs             : {}".format(num_epochs))
    print("  Device             : {}".format(device))
    print("=" * 65)
    print("")

    for epoch in range(num_epochs):
        epoch_start = time.time()
        model.train()
        epoch_loss = 0.0
        all_preds = []
        all_tgts = []

        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            all_preds.append(outputs.detach())
            all_tgts.append(targets)

        epoch_loss /= len(train_loader)
        metrics['loss'].append(epoch_loss)

        all_preds = torch.cat(all_preds, dim=0)
        all_tgts = torch.cat(all_tgts, dim=0)

        psnr, ssim = compute_psnr_ssim(all_preds, all_tgts, device)
        rel_error = compute_relative_error(all_preds, all_tgts)
        metrics['psnr'].append(psnr)
        metrics['ssim'].append(ssim)
        metrics['relative_error'].append(rel_error)

        epoch_time = time.time() - epoch_start
        metrics['time_per_epoch'].append(epoch_time)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                "Epoch {:>3d}/{:d}  |  Loss: {:.6f}  |  "
                "PSNR: {:.2f} dB  |  SSIM: {:.4f}  |  "
                "RE: {:.2f}%  |  Time: {:.2f}s".format(
                    epoch + 1, num_epochs, epoch_loss,
                    psnr, ssim, rel_error, epoch_time
                )
            )

    # Save model weights
    model_path = os.path.join(output_dir, "{}.pth".format(model_name))
    torch.save(model.state_dict(), model_path)
    print("")
    print("Model saved to: {}".format(model_path))

    # Save latent codes for all spatial points
    model.eval()
    with torch.no_grad():
        latent_codes = model.encode(dataset.data.to(device)).cpu()
    latent_path = os.path.join(output_dir, "{}_latent_codes.pt".format(model_name))
    torch.save(latent_codes, latent_path)
    print("Latent codes saved to: {} (shape: {})".format(
        latent_path, list(latent_codes.shape)))

    # Save normalization parameters
    norm_path = os.path.join(output_dir, "{}_normalization.json".format(model_name))
    with open(norm_path, 'w') as f:
        json.dump(dataset.get_normalization_params(), f, indent=2)
    print("Normalization parameters saved to: {}".format(norm_path))

    # Save per-epoch metrics
    metrics_path = os.path.join(output_dir, "{}_metrics.csv".format(model_name))
    df = pd.DataFrame(metrics)
    df['epoch'] = range(1, len(df) + 1)
    df = df[['epoch', 'loss', 'psnr', 'ssim', 'relative_error', 'time_per_epoch']]
    df.to_csv(metrics_path, index=False)
    print("Metrics CSV saved to: {}".format(metrics_path))

    # Print final summary
    total_time = sum(metrics['time_per_epoch'])
    print("")
    print("-" * 65)
    print("  Training completed for: {}".format(model_name))
    print("-" * 65)
    print("  Final Loss           : {:.6f}".format(metrics['loss'][-1]))
    print("  Final PSNR           : {:.2f} dB".format(metrics['psnr'][-1]))
    print("  Final SSIM           : {:.4f}".format(metrics['ssim'][-1]))
    print("  Final Relative Error : {:.2f}%".format(metrics['relative_error'][-1]))
    print("  Total training time  : {:.1f}s".format(total_time))
    print("-" * 65)
    print("")

    return metrics


print("All utilities loaded successfully.")

In [ ]:
# Configuration

DATA_FILE = "/kaggle/input/ml-test-loader-original-data-csv/ML_test_loader_original_data.csv"
EPOCHS = 150
BATCH_SIZE = 512
VIS_TIMESTEP_IDX = 0  # Index of the timestep used for flow-field visualisation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Device        : {}".format(device))
print("Epochs        : {}".format(EPOCHS))
print("Batch size    : {}".format(BATCH_SIZE))

# Load Dataset

dataset = TemporalPointDataset(DATA_FILE)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print("")
print("Visualisation timestep index : {}".format(VIS_TIMESTEP_IDX))
print("Visualisation timestep value : {:.4f}".format(
    dataset.time_values[VIS_TIMESTEP_IDX]))

---
## 1. Base Model: Training

In [ ]:
BASE_OUTPUT_DIR = "/kaggle/working/results/base_autoencoder"

base_model = create_autoencoder('base', dataset.input_dim).to(device)

base_metrics = train_autoencoder(
    model=base_model,
    train_loader=train_loader,
    dataset=dataset,
    device=device,
    num_epochs=EPOCHS,
    model_name='base_linear_ae',
    output_dir=BASE_OUTPUT_DIR,
)

In [ ]:
# Base Model: Training Progress Plots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(base_metrics['loss']) + 1)

axes[0, 0].plot(epochs_range, base_metrics['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, base_metrics['psnr'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].set_title('Peak Signal-to-Noise Ratio')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, base_metrics['ssim'], 'purple', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('SSIM')
axes[1, 0].set_title('Structural Similarity Index')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1.05])

axes[1, 1].plot(epochs_range, base_metrics['relative_error'], 'r-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Relative Error (%)')
axes[1, 1].set_title('Reconstruction Error')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(
    'Base Linear AE (latent dim = 16): Training Progress',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(BASE_OUTPUT_DIR, 'base_ae_training_progress.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

### 1.1 Base Model: Evaluation and Flow-Field Visualisation

In [ ]:
base_model.eval()

total_params = sum(p.numel() for p in base_model.parameters())
print("Evaluating Base model ({:,} parameters)".format(total_params))

with torch.no_grad():
    base_recon, base_latent = base_model(dataset.data.to(device))
    base_recon = base_recon.cpu()
    base_latent = base_latent.cpu()

base_psnr, base_ssim = compute_psnr_ssim(base_recon, dataset.data, device)
base_rel_error = compute_relative_error(base_recon, dataset.data)

print("PSNR           : {:.2f} dB".format(base_psnr))
print("SSIM           : {:.4f}".format(base_ssim))
print("Relative Error : {:.2f}%".format(base_rel_error))

# Reshape reconstructions to (N, T, V) for per-timestep visualisation
base_recon_3d = base_recon.numpy().reshape(
    dataset.num_points, dataset.num_timesteps, dataset.num_vars
)
target_3d = dataset.data.numpy().reshape(
    dataset.num_points, dataset.num_timesteps, dataset.num_vars
)

# Extract a single timestep for flow-field plots
t_idx = VIS_TIMESTEP_IDX
base_pred_t = base_recon_3d[:, t_idx, :]
target_t = target_3d[:, t_idx, :]
base_errors_t = np.abs(target_t - base_pred_t)
x = dataset.coords[:, 0]
y = dataset.coords[:, 1]

print("")
print("Visualising {:,} points at timestep index {} (t = {:.4f})".format(
    len(x), t_idx, dataset.time_values[t_idx]))

In [ ]:
BASE_OUTPUT_DIR = "/kaggle/working/results/base_autoencoder"

# Base Linear AE (latent dim = 16): Flow-Field Visualisation

from matplotlib.gridspec import GridSpec

feature_names = ['Vx', 'Vy', 'Pressure', 'TKE']


fig = plt.figure(figsize=(20, 20))
gs = GridSpec(4, 5, figure=fig,
              width_ratios=[1, 1, 0.05, 1, 0.05],
              wspace=0.35, hspace=0.25)

for row, name in enumerate(feature_names):
    original = target_t[:, row]
    predicted = base_pred_t[:, row]
    error = base_errors_t[:, row]

    ax0 = fig.add_subplot(gs[row, 0])
    ax0.scatter(x, y, c=original, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax0.set_title('Original: {}'.format(name))
    ax0.set_aspect('equal')
    ax0.grid(True, alpha=0.3)

    ax1 = fig.add_subplot(gs[row, 1])
    sc2 = ax1.scatter(x, y, c=predicted, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax1.set_title('Predicted: {}'.format(name))
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)

    cax1 = fig.add_subplot(gs[row, 2])
    fig.colorbar(sc2, cax=cax1)

    ax2 = fig.add_subplot(gs[row, 3])
    sc3 = ax2.scatter(x, y, c=error, cmap='hot', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax2.set_title('Absolute Error: {}'.format(name))
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)

    cax2 = fig.add_subplot(gs[row, 4])
    fig.colorbar(sc3, cax=cax2)

fig.suptitle(
    'Base Linear AE (latent dim = 16): PSNR: {:.2f} dB  |  '
    't = {:.4f}'.format(base_psnr, dataset.time_values[t_idx]),
    fontsize=14, fontweight='bold', y=0.98
)
plt.savefig(
    os.path.join(BASE_OUTPUT_DIR, 'base_ae_flow_visualization.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

print("Base Linear AE (latent dim = 16) evaluation completed.")

---
## 2. Medium Model: Training

In [ ]:
MEDIUM_OUTPUT_DIR = "/kaggle/working/results/medium_autoencoder"

medium_model = create_autoencoder('medium', dataset.input_dim).to(device)

medium_metrics = train_autoencoder(
    model=medium_model,
    train_loader=train_loader,
    dataset=dataset,
    device=device,
    num_epochs=EPOCHS,
    model_name='medium_linear_ae',
    output_dir=MEDIUM_OUTPUT_DIR,
)

In [ ]:
# Medium Model: Training Progress Plots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(medium_metrics['loss']) + 1)

axes[0, 0].plot(epochs_range, medium_metrics['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, medium_metrics['psnr'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].set_title('Peak Signal-to-Noise Ratio')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, medium_metrics['ssim'], 'purple', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('SSIM')
axes[1, 0].set_title('Structural Similarity Index')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1.05])

axes[1, 1].plot(epochs_range, medium_metrics['relative_error'], 'r-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Relative Error (%)')
axes[1, 1].set_title('Reconstruction Error')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(
    'Medium Linear AE (latent dim = 32): Training Progress',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(MEDIUM_OUTPUT_DIR, 'medium_ae_training_progress.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

### 2.1 Medium Model: Evaluation and Flow-Field Visualisation

In [ ]:
medium_model.eval()

total_params = sum(p.numel() for p in medium_model.parameters())
print("Evaluating Medium model ({:,} parameters)".format(total_params))

with torch.no_grad():
    medium_recon, medium_latent = medium_model(dataset.data.to(device))
    medium_recon = medium_recon.cpu()
    medium_latent = medium_latent.cpu()

medium_psnr, medium_ssim = compute_psnr_ssim(medium_recon, dataset.data, device)
medium_rel_error = compute_relative_error(medium_recon, dataset.data)

print("PSNR           : {:.2f} dB".format(medium_psnr))
print("SSIM           : {:.4f}".format(medium_ssim))
print("Relative Error : {:.2f}%".format(medium_rel_error))

medium_recon_3d = medium_recon.numpy().reshape(
    dataset.num_points, dataset.num_timesteps, dataset.num_vars
)
medium_pred_t = medium_recon_3d[:, t_idx, :]
medium_errors_t = np.abs(target_t - medium_pred_t)

In [ ]:
MEDIUM_OUTPUT_DIR = "/kaggle/working/results/medium_autoencoder"

# Medium Linear AE (latent dim = 32): Flow-Field Visualisation

from matplotlib.gridspec import GridSpec

feature_names = ['Vx', 'Vy', 'Pressure', 'TKE']


fig = plt.figure(figsize=(20, 20))
gs = GridSpec(4, 5, figure=fig,
              width_ratios=[1, 1, 0.05, 1, 0.05],
              wspace=0.35, hspace=0.25)

for row, name in enumerate(feature_names):
    original = target_t[:, row]
    predicted = medium_pred_t[:, row]
    error = medium_errors_t[:, row]

    ax0 = fig.add_subplot(gs[row, 0])
    ax0.scatter(x, y, c=original, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax0.set_title('Original: {}'.format(name))
    ax0.set_aspect('equal')
    ax0.grid(True, alpha=0.3)

    ax1 = fig.add_subplot(gs[row, 1])
    sc2 = ax1.scatter(x, y, c=predicted, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax1.set_title('Predicted: {}'.format(name))
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)

    cax1 = fig.add_subplot(gs[row, 2])
    fig.colorbar(sc2, cax=cax1)

    ax2 = fig.add_subplot(gs[row, 3])
    sc3 = ax2.scatter(x, y, c=error, cmap='hot', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax2.set_title('Absolute Error: {}'.format(name))
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)

    cax2 = fig.add_subplot(gs[row, 4])
    fig.colorbar(sc3, cax=cax2)

fig.suptitle(
    'Medium Linear AE (latent dim = 32): PSNR: {:.2f} dB  |  '
    't = {:.4f}'.format(medium_psnr, dataset.time_values[t_idx]),
    fontsize=14, fontweight='bold', y=0.98
)
plt.savefig(
    os.path.join(MEDIUM_OUTPUT_DIR, 'medium_ae_flow_visualization.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

print("Medium Linear AE (latent dim = 32) evaluation completed.")

---
## 3. Large Model: Training

In [ ]:
LARGE_OUTPUT_DIR = "/kaggle/working/results/large_autoencoder"

large_model = create_autoencoder('large', dataset.input_dim).to(device)

large_metrics = train_autoencoder(
    model=large_model,
    train_loader=train_loader,
    dataset=dataset,
    device=device,
    num_epochs=EPOCHS,
    model_name='large_linear_ae',
    output_dir=LARGE_OUTPUT_DIR,
)

In [ ]:
# Large Model: Training Progress Plots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(large_metrics['loss']) + 1)

axes[0, 0].plot(epochs_range, large_metrics['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, large_metrics['psnr'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].set_title('Peak Signal-to-Noise Ratio')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, large_metrics['ssim'], 'purple', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('SSIM')
axes[1, 0].set_title('Structural Similarity Index')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1.05])

axes[1, 1].plot(epochs_range, large_metrics['relative_error'], 'r-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Relative Error (%)')
axes[1, 1].set_title('Reconstruction Error')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(
    'Large Linear AE (latent dim = 64): Training Progress',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(LARGE_OUTPUT_DIR, 'large_ae_training_progress.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

### 3.1 Large Model: Evaluation and Flow-Field Visualisation

In [ ]:
large_model.eval()

total_params = sum(p.numel() for p in large_model.parameters())
print("Evaluating Large model ({:,} parameters)".format(total_params))

with torch.no_grad():
    large_recon, large_latent = large_model(dataset.data.to(device))
    large_recon = large_recon.cpu()
    large_latent = large_latent.cpu()

large_psnr, large_ssim = compute_psnr_ssim(large_recon, dataset.data, device)
large_rel_error = compute_relative_error(large_recon, dataset.data)

print("PSNR           : {:.2f} dB".format(large_psnr))
print("SSIM           : {:.4f}".format(large_ssim))
print("Relative Error : {:.2f}%".format(large_rel_error))

large_recon_3d = large_recon.numpy().reshape(
    dataset.num_points, dataset.num_timesteps, dataset.num_vars
)
large_pred_t = large_recon_3d[:, t_idx, :]
large_errors_t = np.abs(target_t - large_pred_t)

In [ ]:
LARGE_OUTPUT_DIR = "/kaggle/working/results/large_autoencoder"

# Large Linear AE (latent dim = 64): Flow-Field Visualisation

from matplotlib.gridspec import GridSpec

feature_names = ['Vx', 'Vy', 'Pressure', 'TKE']


fig = plt.figure(figsize=(20, 20))
gs = GridSpec(4, 5, figure=fig,
              width_ratios=[1, 1, 0.05, 1, 0.05],
              wspace=0.35, hspace=0.25)

for row, name in enumerate(feature_names):
    original = target_t[:, row]
    predicted = large_pred_t[:, row]
    error = large_errors_t[:, row]

    ax0 = fig.add_subplot(gs[row, 0])
    ax0.scatter(x, y, c=original, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax0.set_title('Original: {}'.format(name))
    ax0.set_aspect('equal')
    ax0.grid(True, alpha=0.3)

    ax1 = fig.add_subplot(gs[row, 1])
    sc2 = ax1.scatter(x, y, c=predicted, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax1.set_title('Predicted: {}'.format(name))
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)

    cax1 = fig.add_subplot(gs[row, 2])
    fig.colorbar(sc2, cax=cax1)

    ax2 = fig.add_subplot(gs[row, 3])
    sc3 = ax2.scatter(x, y, c=error, cmap='hot', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax2.set_title('Absolute Error: {}'.format(name))
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)

    cax2 = fig.add_subplot(gs[row, 4])
    fig.colorbar(sc3, cax=cax2)

fig.suptitle(
    'Large Linear AE (latent dim = 64): PSNR: {:.2f} dB  |  '
    't = {:.4f}'.format(large_psnr, dataset.time_values[t_idx]),
    fontsize=14, fontweight='bold', y=0.98
)
plt.savefig(
    os.path.join(LARGE_OUTPUT_DIR, 'large_ae_flow_visualization.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

print("Large Linear AE (latent dim = 64) evaluation completed.")

---
## 4. Combined Comparison: All Three Models

In [ ]:
# Training Curves Comparison

COMPARISON_DIR = "/kaggle/working/results/comparison_linear_ae"
os.makedirs(COMPARISON_DIR, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

models_data = [
    ('Base (latent = 16)',   base_metrics,   'tab:blue'),
    ('Medium (latent = 32)', medium_metrics, 'tab:orange'),
    ('Large (latent = 64)',  large_metrics,  'tab:green'),
]

for label, m, color in models_data:
    er = range(1, len(m['loss']) + 1)
    axes[0, 0].plot(er, m['loss'],           '-', color=color, linewidth=2, label=label)
    axes[0, 1].plot(er, m['psnr'],           '-', color=color, linewidth=2, label=label)
    axes[1, 0].plot(er, m['ssim'],           '-', color=color, linewidth=2, label=label)
    axes[1, 1].plot(er, m['relative_error'], '-', color=color, linewidth=2, label=label)

axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].set_title('Peak Signal-to-Noise Ratio')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('SSIM')
axes[1, 0].set_title('Structural Similarity Index')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1.05])
axes[1, 0].legend()

axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Relative Error (%)')
axes[1, 1].set_title('Reconstruction Error')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.suptitle(
    'Linear AutoEncoder: Model Comparison (Training)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_DIR, 'ae_training_comparison.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# Evaluation Metrics: Bar Charts

model_labels = [
    'Base\n(latent = 16)',
    'Medium\n(latent = 32)',
    'Large\n(latent = 64)',
]
eval_psnrs  = [base_psnr,      medium_psnr,      large_psnr]
eval_ssims  = [base_ssim,      medium_ssim,      large_ssim]
eval_errors = [base_rel_error, medium_rel_error, large_rel_error]
colors = ['tab:blue', 'tab:orange', 'tab:green']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# PSNR
bars1 = axes[0].bar(model_labels, eval_psnrs, color=colors,
                     edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('Evaluation PSNR')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, eval_psnrs):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        '{:.2f}'.format(val), ha='center', va='bottom', fontsize=10
    )

# SSIM
bars2 = axes[1].bar(model_labels, eval_ssims, color=colors,
                     edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('SSIM')
axes[1].set_title('Evaluation SSIM')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, 1.1])
for bar, val in zip(bars2, eval_ssims):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
        '{:.4f}'.format(val), ha='center', va='bottom', fontsize=10
    )

# Relative Error
bars3 = axes[2].bar(model_labels, eval_errors, color=colors,
                     edgecolor='black', linewidth=0.5)
axes[2].set_ylabel('Relative Error (%)')
axes[2].set_title('Evaluation Relative Error')
axes[2].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars3, eval_errors):
    axes[2].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
        '{:.2f}%'.format(val), ha='center', va='bottom', fontsize=10
    )

plt.suptitle(
    'Linear AutoEncoder: Evaluation Metrics Comparison',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_DIR, 'ae_evaluation_comparison.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# Compression Analysis: Storage Breakdown and Ratios

comp_base   = compute_compression_ratio(base_model, dataset)
comp_medium = compute_compression_ratio(medium_model, dataset)
comp_large  = compute_compression_ratio(large_model, dataset)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar chart: model weights + latent codes
model_sizes_mb = [
    comp_base['model_size_kb'] / 1024,
    comp_medium['model_size_kb'] / 1024,
    comp_large['model_size_kb'] / 1024,
]
latent_sizes_mb = [
    comp_base['latent_size_kb'] / 1024,
    comp_medium['latent_size_kb'] / 1024,
    comp_large['latent_size_kb'] / 1024,
]
x_pos = range(len(model_labels))

axes[0].bar(
    x_pos, model_sizes_mb, color='steelblue',
    label='Model Weights', edgecolor='black', linewidth=0.5
)
axes[0].bar(
    x_pos, latent_sizes_mb, bottom=model_sizes_mb, color='coral',
    label='Latent Codes', edgecolor='black', linewidth=0.5
)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_labels)
axes[0].set_ylabel('Size (MB)')
axes[0].set_title('Compressed Storage Breakdown')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
for i, (ms, ls) in enumerate(zip(model_sizes_mb, latent_sizes_mb)):
    axes[0].text(
        i, ms + ls + 0.2, '{:.1f} MB'.format(ms + ls),
        ha='center', fontsize=10
    )

# Compression ratio bars
ratios = [
    comp_base['compression_ratio'],
    comp_medium['compression_ratio'],
    comp_large['compression_ratio'],
]
bars_r = axes[1].bar(
    model_labels, ratios, color=colors,
    edgecolor='black', linewidth=0.5
)
axes[1].set_ylabel('Compression Ratio')
axes[1].set_title('Compression Ratio (higher is better)')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars_r, ratios):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
        '{:.1f}:1'.format(val), ha='center', va='bottom', fontsize=10
    )

plt.suptitle(
    'Linear AutoEncoder: Compression Analysis',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_DIR, 'ae_compression_comparison.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

---
## 5. Latent Space Analysis

In [ ]:
# Latent Space: Value Distribution

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

latent_info = [
    ('Base (dim = 16)',   base_latent,   'tab:blue'),
    ('Medium (dim = 32)', medium_latent, 'tab:orange'),
    ('Large (dim = 64)',  large_latent,  'tab:green'),
]

for ax, (name, latent, color) in zip(axes, latent_info):
    values = latent.numpy().flatten()
    ax.hist(values, bins=100, color=color, alpha=0.7, density=True)
    ax.set_xlabel('Latent Value')
    ax.set_ylabel('Density')
    ax.set_title('{}\nmean = {:.3f}, std = {:.3f}'.format(
        name, latent.mean().item(), latent.std().item()))
    ax.grid(True, alpha=0.3)

plt.suptitle(
    'Latent Space Value Distribution',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_DIR, 'ae_latent_distribution.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# Latent Space: Per-Dimension Variance

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, latent, color) in zip(axes, latent_info):
    variances = latent.var(dim=0).numpy()
    ax.bar(range(len(variances)), variances, color=color, alpha=0.8)
    ax.set_xlabel('Latent Dimension')
    ax.set_ylabel('Variance')
    ax.set_title(name)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    'Per-Dimension Variance in Latent Space (higher = more informative)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_DIR, 'ae_latent_variance.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# Latent Space: Spatial Map of First Two Dimensions

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

spatial_info = [
    ('Base',   base_latent),
    ('Medium', medium_latent),
    ('Large',  large_latent),
]

for row, (name, latent) in enumerate(spatial_info):
    for col in range(2):
        sc = axes[row, col].scatter(
            x, y, c=latent[:, col].numpy(),
            cmap='coolwarm', s=0.5, alpha=0.8
        )
        axes[row, col].set_title(
            '{}: Latent Dimension {}'.format(name, col)
        )
        axes[row, col].set_aspect('equal')
        axes[row, col].grid(True, alpha=0.3)
        plt.colorbar(sc, ax=axes[row, col])

plt.suptitle(
    'Spatial Structure in Latent Space (first two dimensions)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_DIR, 'ae_latent_spatial_map.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

---
## 6. Per-Timestep Reconstruction Quality

In [ ]:
# Per-Timestep PSNR and Relative Error

print("Computing per-timestep metrics for all three models...")

timestep_psnrs  = {'base': [], 'medium': [], 'large': []}
timestep_errors = {'base': [], 'medium': [], 'large': []}

recon_3d_map = {
    'base':   base_recon_3d,
    'medium': medium_recon_3d,
    'large':  large_recon_3d,
}

for t_i in range(dataset.num_timesteps):
    target_ti = dataset.data_3d[:, t_i, :]

    for name in ['base', 'medium', 'large']:
        pred_ti = torch.FloatTensor(recon_3d_map[name][:, t_i, :])

        psnr_metric = PeakSignalNoiseRatio()
        psnr_metric.update(pred_ti, target_ti)
        timestep_psnrs[name].append(psnr_metric.compute().item())

        err = torch.norm(pred_ti - target_ti) / torch.norm(target_ti) * 100
        timestep_errors[name].append(err.item())

print("Done. Computed metrics across {} timesteps.".format(dataset.num_timesteps))

# Plot
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

timesteps = range(dataset.num_timesteps)
plot_info = [
    ('base',   'Base (latent = 16)',   'tab:blue'),
    ('medium', 'Medium (latent = 32)', 'tab:orange'),
    ('large',  'Large (latent = 64)',  'tab:green'),
]

for key, label, color in plot_info:
    axes[0].plot(
        timesteps, timestep_psnrs[key], '-',
        color=color, linewidth=1.5, label=label, alpha=0.8
    )
    axes[1].plot(
        timesteps, timestep_errors[key], '-',
        color=color, linewidth=1.5, label=label, alpha=0.8
    )

axes[0].set_xlabel('Timestep Index')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('Per-Timestep PSNR')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel('Timestep Index')
axes[1].set_ylabel('Relative Error (%)')
axes[1].set_title('Per-Timestep Relative Error')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle(
    'Linear AutoEncoder: Reconstruction Quality Across Timesteps',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_DIR, 'ae_per_timestep_quality.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

---
## 7. Summary

In [ ]:
# Final Summary Table

header = "{:<30s} {:>18s} {:>20s} {:>20s}".format(
    "Metric", "Base (latent=16)", "Medium (latent=32)", "Large (latent=64)"
)

print("")
print("=" * 90)
print("  LINEAR AUTOENCODER: THREE-MODEL COMPARISON SUMMARY")
print("=" * 90)
print(header)
print("-" * 90)

print("{:<30s} {:>18,d} {:>20,d} {:>20,d}".format(
    "Parameters",
    comp_base['total_params'],
    comp_medium['total_params'],
    comp_large['total_params'],
))
print("{:<30s} {:>18.1f} {:>20.1f} {:>20.1f}".format(
    "Model Size (KB)",
    comp_base['model_size_kb'],
    comp_medium['model_size_kb'],
    comp_large['model_size_kb'],
))
print("{:<30s} {:>18.1f} {:>20.1f} {:>20.1f}".format(
    "Latent Codes (KB)",
    comp_base['latent_size_kb'],
    comp_medium['latent_size_kb'],
    comp_large['latent_size_kb'],
))
print("{:<30s} {:>18.1f} {:>20.1f} {:>20.1f}".format(
    "Total Compressed (KB)",
    comp_base['compressed_size_kb'],
    comp_medium['compressed_size_kb'],
    comp_large['compressed_size_kb'],
))
print("{:<30s} {:>17.1f}:1 {:>19.1f}:1 {:>19.1f}:1".format(
    "Compression Ratio",
    comp_base['compression_ratio'],
    comp_medium['compression_ratio'],
    comp_large['compression_ratio'],
))

print("-" * 90)

print("{:<30s} {:>18.6f} {:>20.6f} {:>20.6f}".format(
    "Final Training Loss",
    base_metrics['loss'][-1],
    medium_metrics['loss'][-1],
    large_metrics['loss'][-1],
))
print("{:<30s} {:>18.2f} {:>20.2f} {:>20.2f}".format(
    "Final Training PSNR (dB)",
    base_metrics['psnr'][-1],
    medium_metrics['psnr'][-1],
    large_metrics['psnr'][-1],
))
print("{:<30s} {:>18.4f} {:>20.4f} {:>20.4f}".format(
    "Final Training SSIM",
    base_metrics['ssim'][-1],
    medium_metrics['ssim'][-1],
    large_metrics['ssim'][-1],
))
print("{:<30s} {:>18.2f} {:>20.2f} {:>20.2f}".format(
    "Final Relative Error (%)",
    base_metrics['relative_error'][-1],
    medium_metrics['relative_error'][-1],
    large_metrics['relative_error'][-1],
))

print("-" * 90)

print("{:<30s} {:>18.2f} {:>20.2f} {:>20.2f}".format(
    "Eval PSNR (dB)",
    base_psnr, medium_psnr, large_psnr,
))
print("{:<30s} {:>18.4f} {:>20.4f} {:>20.4f}".format(
    "Eval SSIM",
    base_ssim, medium_ssim, large_ssim,
))
print("{:<30s} {:>18.2f} {:>20.2f} {:>20.2f}".format(
    "Eval Relative Error (%)",
    base_rel_error, medium_rel_error, large_rel_error,
))
print("{:<30s} {:>18.1f} {:>20.1f} {:>20.1f}".format(
    "Total Training Time (s)",
    sum(base_metrics['time_per_epoch']),
    sum(medium_metrics['time_per_epoch']),
    sum(large_metrics['time_per_epoch']),
))

print("=" * 90)

In [ ]:
# Save All Evaluation Results to JSON

all_results = {
    'base': {
        'architecture': '1200-256-128-16-128-256-1200',
        'latent_dim': 16,
        'parameters': comp_base['total_params'],
        'model_size_kb': comp_base['model_size_kb'],
        'latent_size_kb': comp_base['latent_size_kb'],
        'compressed_size_kb': comp_base['compressed_size_kb'],
        'compression_ratio': comp_base['compression_ratio'],
        'eval_psnr': float(base_psnr),
        'eval_ssim': float(base_ssim),
        'eval_relative_error': float(base_rel_error),
        'training_time_s': sum(base_metrics['time_per_epoch']),
    },
    'medium': {
        'architecture': '1200-512-256-128-32-128-256-512-1200',
        'latent_dim': 32,
        'parameters': comp_medium['total_params'],
        'model_size_kb': comp_medium['model_size_kb'],
        'latent_size_kb': comp_medium['latent_size_kb'],
        'compressed_size_kb': comp_medium['compressed_size_kb'],
        'compression_ratio': comp_medium['compression_ratio'],
        'eval_psnr': float(medium_psnr),
        'eval_ssim': float(medium_ssim),
        'eval_relative_error': float(medium_rel_error),
        'training_time_s': sum(medium_metrics['time_per_epoch']),
    },
    'large': {
        'architecture': '1200-512-256-128-64-128-256-512-1200',
        'latent_dim': 64,
        'parameters': comp_large['total_params'],
        'model_size_kb': comp_large['model_size_kb'],
        'latent_size_kb': comp_large['latent_size_kb'],
        'compressed_size_kb': comp_large['compressed_size_kb'],
        'compression_ratio': comp_large['compression_ratio'],
        'eval_psnr': float(large_psnr),
        'eval_ssim': float(large_ssim),
        'eval_relative_error': float(large_rel_error),
        'training_time_s': sum(large_metrics['time_per_epoch']),
    },
}

results_path = os.path.join(COMPARISON_DIR, 'linear_ae_all_results.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print("All results saved to: {}".format(results_path))